# Master Import — multilingual legal corpus

Single orchestrator for the four legal sources. Each source is scraped/imported
**independently into its own file** in `../data/` — nothing is merged across sources.
This notebook does **scraping + metadata capture + new/extend merge only**.
No chunking, embedding, KWIC or other NLP (those are downstream steps).

| source    | jurisdiction       | endpoint                          | stable id   | output file                       |
|-----------|--------------------|-----------------------------------|-------------|-----------------------------------|
| `ris`     | Austria (AT)       | RIS OGD API v2.6 (Judikatur)      | `id`        | `ris_<topic>.json`                |
| `echr`    | Council of Europe  | HUDOC via `echr_extractor`        | `itemid`    | `echr_<topic>.json`               |
| `swiss`   | Switzerland (CH)   | entscheidsuche.ch (Elasticsearch) | `Signatur`  | `swiss_<topic>.json`              |
| `germany` | Germany (DE)       | de.openlegaldata.io API           | `slug`      | `germany_<topic>.json`            |

**RUN_MODE**
- `"new"`  — fresh import for a topic. If the per-source file already exists it is moved to `../archive/` first (never overwritten), then re-imported.
- `"extend"` — load each existing per-source file, fetch the requested delta (new `KEYWORDS` and/or wider `YEAR_FROM`/`YEAR_TO`), and merge by stable id. Idempotent: re-running an overlapping keyword/year overwrites identical records instead of duplicating them, and accumulates `matched_keywords`. Errors clearly if no existing file is found.

## 1. Configuration — the only cell you normally edit

In [ ]:
# ---- run parameters -------------------------------------------------------
RUN_MODE  = "new"                 # "new" (fresh topic) or "extend" (add to existing topic)
TOPIC     = "parental_alienation" # identifier used in every output filename

# Per-source search terms — alienation/child-welfare specific, in each system's
# vocabulary. Broad terms (Kindeswohl, Obhut, Umgangsrecht, elterliche Sorge) were
# dropped: they appear across migration/admin/social courts and flood the corpus.
# In "extend" mode, add new terms under the relevant source.
KEYWORDS = {
    "ris": [        # Austria
        "Entfremdung", "elterliche Entfremdung", "Eltern-Kind-Entfremdung",
        "Kindeswohlgefährdung", "Loyalitätskonflikt", "Kontaktverweigerung",
    ],
    "echr": [       # Council of Europe — English (HUDOC ENG)
        "parental alienation", "best interests of the child",
        "contact rights", "child welfare",
    ],
    "swiss": [      # Switzerland
        "Entfremdung", "elterliche Entfremdung", "Eltern-Kind-Entfremdung",
        "Kindeswohlgefährdung", "Loyalitätskonflikt", "Kontaktverweigerung",
    ],
    "germany": [    # Germany
        "Entfremdung", "elterliche Entfremdung", "Eltern-Kind-Entfremdung",
        "Kindeswohlgefährdung", "Loyalitätskonflikt",
        "Umgangsverweigerung", "Umgangsvereitelung",
    ],
}

YEAR_FROM = 2000                  # inclusive start year (extended from 2015 on 2026-07-07)
YEAR_TO   = 2025                  # inclusive end year
SOURCES   = ["ris", "echr", "swiss"]   # subset to run only some , "germany"
FETCH_FULL_TEXT = True            # False = metadata only (much faster)

# ---- family / civil court filtering ---------------------------------------
# Drop administrative / social-insurance / migration / tax / federal-criminal
# courts so the corpus stays family-law. Set the flags False to keep everything.
SWISS_FAMILY_COURTS_ONLY = True
# Swiss court-type suffixes to drop: Verwaltungsgericht (VG), Bundesverwaltungs-
# gericht (BVGE), Sozialversicherungsgericht (SVG/VSG/EVG), Bundesstrafgericht
# (BSTG), Patentgericht (PATG), Departement/Regierungsrat (DEP/RR), VWEK.
SWISS_EXCLUDE_COURTS = {"VG", "VGR", "VGER", "BVGE", "BVGER", "SVG", "VSG", "EVG",
                        "BSTG", "BSTGER", "PATG", "DEP", "RR", "VWEK"}
DE_ORDINARY_ONLY = True
DE_EXCLUDE_JURIS = {"Verwaltungsgerichtsbarkeit", "Sozialgerichtsbarkeit",
                    "Finanzgerichtsbarkeit", "Arbeitsgerichtsbarkeit",
                    "Verfassungsgerichtsbarkeit"}

# ---- RIS decision expansion -----------------------------------------------
# RIS Justiz search only returns Rechtssätze (legal principles). The actual court
# decisions (Dokumenttyp "Text") are reached only via each Rechtssatz's linked
# Entscheidungstexte and fetched from their document page. With this on, those
# decisions are imported as their own records alongside the Rechtssätze.
RIS_FETCH_DECISIONS = True
RIS_DECISION_SCOPE  = "year_range"  # "year_range" (within YEAR_FROM-YEAR_TO) | "all" | "leading"

# ---- RIS full-text (Entscheidungstext) search ------------------------------
# The API default searches Rechtssatz headnotes only, so the RS-anchored route above
# misses decisions whose headnote does not carry the phrase, and decisions with no
# Rechtssatz at all. Measured recall gap: reports/ris_recall_probe.md. This second
# pass searches the decision texts directly.
# Suchworte is a PHRASE match, so the Austrian caption convention
# ("Pflegschaftssache ... wegen Obsorge") works as a precision filter: bare "Obsorge"
# also matches the party description of any proceeding involving a minor
# ("beide in Obsorge der Mutter"), which is boilerplate, not subject matter.
RIS_FULLTEXT_SEARCH   = True
RIS_FULLTEXT_KEYWORDS = [
    "wegen Obsorge", "wegen Kontaktrecht", "wegen Besuchsrecht",   # caption-anchored
    "Kindeswohlgefährdung", "Loyalitätskonflikt",                  # topic-specific
    "elterliche Entfremdung", "Eltern-Kind-Entfremdung", "Kontaktverweigerung",
]
# "Besuchsrecht" was renamed "Kontaktrecht" by the KindNamRÄG 2013 — caption hits split
# 201 vs 1 before 2013 and 88 vs 424 after, so both terms are needed to cover 2000-2025.
# Bare "Entfremdung" is deliberately NOT in this list: in the decision texts it is
# dominated by the criminal-senate homonym (misappropriation — 389 of 510 hits) and by
# civil senses unrelated to children (spousal estrangement, insurance). It stays on the
# Rechtssatz route above, where the headnote vocabulary keeps it family-law.
RIS_EXCLUDE_RECHTSGEBIETE  = {"strafrecht"}   # the Entfremdung homonym
RIS_EXCLUDE_GERICHT_PREFIX = ("AUSL",)        # EGMR summaries duplicate the ECHR corpus

# ---- politeness / paging tuning (rarely changed) --------------------------
REQUEST_DELAY = 1.5    # seconds between HTTP requests
ECHR_COUNT    = 5000   # max cases per ECHR keyword query
SWISS_MAX     = 10000  # Elasticsearch from+size ceiling per keyword
DE_MAX_PAGES  = 50     # max search pages per German keyword (10 results/page)

assert RUN_MODE in ("new", "extend"), "RUN_MODE must be 'new' or 'extend'"
assert isinstance(KEYWORDS, dict), "KEYWORDS must be a dict keyed by source"
assert any(KEYWORDS.get(s) for s in SOURCES), "no keywords for any selected source"
assert YEAR_FROM <= YEAR_TO, "YEAR_FROM must be <= YEAR_TO"
assert RIS_DECISION_SCOPE in ("year_range", "all", "leading")
assert isinstance(RIS_FULLTEXT_KEYWORDS, list)
print(f"mode={RUN_MODE}  topic={TOPIC}  years={YEAR_FROM}-{YEAR_TO}  sources={SOURCES}")
for _s in SOURCES:
    print(f"  {_s:8s} keywords: {KEYWORDS.get(_s, [])}")


mode=new  topic=parental_alienation  years=2000-2025  sources=['ris', 'echr', 'swiss']
  ris      keywords: ['Entfremdung', 'elterliche Entfremdung', 'Eltern-Kind-Entfremdung', 'Kindeswohlgefährdung', 'Loyalitätskonflikt', 'Kontaktverweigerung']
  echr     keywords: ['parental alienation', 'best interests of the child', 'contact rights', 'child welfare']
  swiss    keywords: ['Entfremdung', 'elterliche Entfremdung', 'Eltern-Kind-Entfremdung', 'Kindeswohlgefährdung', 'Loyalitätskonflikt', 'Kontaktverweigerung']


## 2. Imports & shared helpers
Provenance, archive-on-new, load/save, and the idempotent merge live here and are
reused by every source.

In [ ]:
import json
import time
import logging
from datetime import datetime, date
from pathlib import Path
from xml.etree import ElementTree as ET

import requests
from bs4 import BeautifulSoup

DATA_DIR    = Path("../data")
ARCHIVE_DIR = Path("../archive")

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)-7s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("master_import")

session = requests.Session()
session.headers.update({
    "User-Agent": "MasterThesis-Research/1.0 (Academic; msmirnov98@gmail.com)",
    "Accept": "application/json",
})


def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def today_str():
    return date.today().isoformat()


def html_to_text(html):
    # strip HTML to plain text; html.parser (no lxml dependency)
    if not html:
        return ""
    return BeautifulSoup(html, "html.parser").get_text(separator="\n", strip=True)


def out_path(source):
    # flat per-source file in data/, e.g. data/ris_<topic>.json
    return DATA_DIR / f"{source}_{TOPIC}.json"


def archive_if_exists(path):
    # move an existing per-source file into archive/ with a timestamp suffix
    if path.exists():
        ARCHIVE_DIR.mkdir(exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest = ARCHIVE_DIR / f"{path.stem}_{stamp}{path.suffix}"
        path.rename(dest)
        log.info(f"Archived existing {path.name} -> {dest.name}")
        return dest
    return None


def load_records(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_records(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=1)
    log.info(f"Saved {len(records)} records -> {path}")


def add_provenance(rec, source, jurisdiction, lang, source_url, stable_id):
    # uniform provenance block on every record across all sources
    rec["stable_id"]      = stable_id
    rec["source"]         = source
    rec["jurisdiction"]   = jurisdiction
    rec["lang"]           = lang
    rec["source_url"]     = source_url
    rec["retrieved_date"] = today_str()
    return rec


# list-valued provenance fields: unioned on merge instead of overwritten, so a record
# found again by a different route/keyword accumulates its labels.
UNION_FIELDS = ("matched_keywords", "match_route", "from_rechtssatz")


def merge_by_id(existing, fetched, id_key="stable_id", union_fields=UNION_FIELDS):
    # idempotent merge keyed on stable id: a freshly fetched record overwrites the
    # existing one with the same id, except that the union fields accumulate and a
    # non-empty existing full_text is never replaced by an empty re-fetch.
    merged = {}
    for r in existing:
        k = r.get(id_key)
        if k is not None:
            merged[k] = r
    added = 0
    for r in fetched:
        k = r.get(id_key)
        if k is None:
            continue
        if k in merged:
            prev = merged[k]
            for f in union_fields:
                vals = set(prev.get(f) or []) | set(r.get(f) or [])
                if vals:
                    r[f] = sorted(vals)
            if not r.get("full_text") and prev.get("full_text"):
                r["full_text"]   = prev["full_text"]
                r["text_length"] = prev.get("text_length", len(prev["full_text"]))
        else:
            added += 1
        merged[k] = r
    log.info(f"merge on '{id_key}': {len(existing)} existing + {len(fetched)} fetched "
             f"-> {len(merged)} total ({added} new)")
    return list(merged.values())


print("helpers ready")

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
helpers ready


## 3. RIS (Austria) — RIS OGD API v2.6
Two search routes, both paged with the documented v2.6 parameters
(`DokumenteProSeite` / `Seitennummer` — an earlier note here blamed broken `Seite`
pagination, which was a parameter-name bug; no year-windowing is needed):

1. **Rechtssätze** (API default) — distilled legal principles, plus the court decisions
   reached through each Rechtssatz's linked **Entscheidungstexte**.
2. **Entscheidungstexte** (`Dokumenttyp.SucheInEntscheidungstexten=true`) — the decision
   texts searched directly, which is the only way to reach decisions with no Rechtssatz
   or whose headnote does not carry the phrase. Criminal senates (the *Entfremdung*
   homonym) and AUSL EGMR summaries are filtered out.

Every record carries `match_route` so the two routes stay separable downstream.
Captures norm references, Geschäftszahl, decision date, Rechtssatznummern,
Schlagworte/Beisätze when present, and the full Entscheidungstexte list.

In [ ]:
RIS_API = "https://data.bka.gv.at/ris/api/v2.6/Judikatur"
RIS_APP = "Justiz"   # OGH/OLG/LG/BG — family-law relevant courts

from urllib.parse import urlparse, parse_qs


def ris_get(params, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.get(RIS_API, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.HTTPError:
            code = r.status_code
            if code == 429:
                time.sleep(10 * (attempt + 1))
            elif code >= 500:
                time.sleep(5 * (attempt + 1))
            else:
                return None
        except (requests.exceptions.RequestException, ValueError):
            time.sleep(3 * (attempt + 1))
    return None


def ris_fetch_text(url, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.get(url, timeout=30)
            r.raise_for_status()
            r.encoding = "utf-8"
            return r.text
        except Exception:
            time.sleep(3 * (attempt + 1))
    return None


def ris_parse(data):
    if not data:
        return 0, []
    try:
        res = data["OgdSearchResult"]["OgdDocumentResults"]
        total = int(res.get("Hits", {}).get("#text", 0))
        docs = res.get("OgdDocumentReference", [])
        if isinstance(docs, dict):   # single result comes as dict, not list
            docs = [docs]
        return total, docs
    except (KeyError, TypeError):
        return 0, []


RIS_PAGE_SIZE   = "OneHundred"   # DokumenteProSeite: Ten | Twenty | Fifty | OneHundred
RIS_PAGE_SIZE_N = 100
RIS_MAX_PAGES   = 80             # safety ceiling; 8000 documents per query


def _ris_doc_id(doc):
    return doc.get("Data", {}).get("Metadaten", {}).get("Technisch", {}).get("ID", "")


def ris_search(params):
    # page through a query until every reported Hit is retrieved. The v2.6 paging
    # parameters are DokumenteProSeite/Seitennummer; a page past the end of the result
    # set answers HTTP 500, which ris_get maps to None -> loop stops.
    out, seen, hits = [], set(), None
    for pg in range(1, RIS_MAX_PAGES + 1):
        total, page = ris_parse(ris_get({**params,
                                         "DokumenteProSeite": RIS_PAGE_SIZE,
                                         "Seitennummer": pg}))
        if hits is None:
            hits = total
        if not page:
            break
        for doc in page:
            did = _ris_doc_id(doc)
            if did and did not in seen:
                seen.add(did)
                out.append(doc)
        if len(page) < RIS_PAGE_SIZE_N or len(seen) >= hits:
            break
    if hits and len(seen) < hits:
        log.warning(f"[RIS] retrieved {len(seen)}/{hits} for {params.get('Suchworte')!r}")
    return hits or 0, out


def _ris_item(d, key):
    # a ".item" field that may be a str, a {"item": ...} dict, or a list
    val = d.get(key, {})
    if isinstance(val, str):
        return val
    if isinstance(val, dict):
        item = val.get("item", "")
        if isinstance(item, list):
            return "; ".join(str(i) for i in item)
        return str(item) if item else ""
    if isinstance(val, list):
        return "; ".join(str(i) for i in val)
    return ""


def ris_extract(doc):
    data   = doc.get("Data", {})
    meta   = data.get("Metadaten", {})
    tech   = meta.get("Technisch", {})
    allg   = meta.get("Allgemein", {})
    jud    = meta.get("Judikatur", {})
    justiz = jud.get("Justiz", {})

    # content URLs for full-text fetching
    content_urls = {}
    try:
        cref = data.get("Dokumentliste", {}).get("ContentReference", {})
        if isinstance(cref, list):
            cref = cref[0] if cref else {}
        ulist = cref.get("Urls", {}).get("ContentUrl", [])
        if isinstance(ulist, dict):
            ulist = [ulist]
        for u in ulist:
            if isinstance(u, dict):
                content_urls[u.get("DataType", "")] = u.get("Url", "")
    except (KeyError, TypeError, AttributeError):
        pass

    # Entscheidungstexte = the linked decisions / case-citation list (keep full items)
    et_items = []
    et_raw = justiz.get("Entscheidungstexte", {})
    if isinstance(et_raw, dict):
        items = et_raw.get("item", [])
        if isinstance(items, dict):
            items = [items]
        if isinstance(items, list):
            et_items = items

    return {
        "id":                 tech.get("ID", ""),
        "applikation":        tech.get("Applikation", ""),
        "organ":              tech.get("Organ", ""),
        "gericht":            justiz.get("Gericht", ""),
        "dokumenttyp":        jud.get("Dokumenttyp", ""),
        "geschaeftszahl":     _ris_item(jud, "Geschaeftszahl"),
        "normen":             _ris_item(jud, "Normen"),
        "entscheidungsdatum": jud.get("Entscheidungsdatum", ""),
        "ecli":               jud.get("EuropeanCaseLawIdentifier", ""),
        "rechtsgebiete":      _ris_item(justiz, "Rechtsgebiete"),
        "rechtssatznummern":  _ris_item(justiz, "Rechtssatznummern"),
        "schlagworte":        _ris_item(justiz, "Schlagworte") or _ris_item(jud, "Schlagworte"),
        "beisaetze":          _ris_item(justiz, "Beisaetze") or _ris_item(jud, "Beisaetze"),
        "entscheidungstexte": et_items,
        "n_entscheidungstexte": len(et_items),
        "dokument_url":       allg.get("DokumentUrl", ""),
        "veroeffentlicht":    allg.get("Veroeffentlicht", ""),
        "geaendert":          allg.get("Geaendert", ""),
        "content_url_html":   content_urls.get("Html", ""),
        "content_url_xml":    content_urls.get("Xml", ""),
        "content_url_pdf":    content_urls.get("Pdf", ""),
    }


def ris_full_text(rec):
    # strategy: HTML content url -> XML content url -> RIS page fallback
    for url in [rec.get("content_url_html", ""), rec.get("content_url_xml", "")]:
        if not url:
            continue
        raw = ris_fetch_text(url)
        if not raw:
            continue
        if "Xml" in url or url.endswith(".xml"):
            try:
                root = ET.fromstring(raw)
                parts = [t.strip() for el in root.iter()
                         for t in (el.text, el.tail) if t and t.strip()]
                txt = "\n".join(parts)
                if len(txt) > 50:
                    return txt
            except ET.ParseError:
                pass
        txt = html_to_text(raw)
        if len(txt) > 50:
            return txt
    dok = rec.get("dokument_url", "")
    if dok:
        raw = ris_fetch_text(dok)
        if raw:
            return html_to_text(raw)
    return ""


def _ris_year(rec):
    try:
        return int(str(rec.get("entscheidungsdatum", ""))[:4])
    except (ValueError, TypeError):
        return None


# ---- linked-decision expansion (Entscheidungstexte -> full "Text" records) ----

def _ris_dn_from_url(url):
    # pull the Dokumentnummer out of a .wxe DokumentUrl
    try:
        return parse_qs(urlparse(url).query).get("Dokumentnummer", [""])[0]
    except Exception:
        return ""


def _ris_item_year(it):
    try:
        return int(str(it.get("Entscheidungsdatum", ""))[:4])
    except (ValueError, TypeError):
        return None


def ris_decision_text(dn):
    # decisions aren't served by the API; fetch the clean content URL by Dokumentnummer
    clean = f"https://www.ris.bka.gv.at/Dokumente/Justiz/{dn}/{dn}.html"
    raw = ris_fetch_text(clean)
    if raw:
        txt = html_to_text(raw)
        if len(txt) > 50:
            return txt
    return ""


def ris_collect_decision_stubs(rs_records, year_from, year_to, scope):
    # gather unique linked decisions from the Rechtssatz records, per scope.
    # returns {dokumentnummer: {"stub": item, "from_rs": set, "keywords": set}}
    stubs = {}
    for rs in rs_records:
        if rs.get("dokumenttyp") != "Rechtssatz":
            continue
        rs_no = rs.get("rechtssatznummern") or rs.get("id")
        rs_kw = rs.get("matched_keywords") or []
        items = rs.get("entscheidungstexte") or []
        if scope == "leading":
            rs_year = _ris_year(rs)
            chosen = []
            for it in items:
                if it.get("DokumentUrl") and _ris_item_year(it) == rs_year:
                    chosen = [it]
                    break
            if not chosen:
                for it in items:
                    if it.get("DokumentUrl"):
                        chosen = [it]
                        break
        else:
            chosen = []
            for it in items:
                if not it.get("DokumentUrl"):
                    continue
                if scope == "year_range":
                    y = _ris_item_year(it)
                    if y is None or not (year_from <= y <= year_to):
                        continue
                chosen.append(it)
        for it in chosen:
            dn = _ris_dn_from_url(it.get("DokumentUrl", ""))
            if not dn:
                continue
            e = stubs.setdefault(dn, {"stub": it, "from_rs": set(), "keywords": set()})
            if rs_no:
                e["from_rs"].add(rs_no)
            e["keywords"].update(rs_kw)
    return stubs


def ris_decision_record(dn, info, cached_text=None):
    it = info["stub"]
    txt = cached_text or (ris_decision_text(dn) if FETCH_FULL_TEXT else "")
    url = it.get("DokumentUrl", "")
    rec = {
        "id":                 dn,
        "applikation":        "Justiz",
        "organ":              it.get("Gericht", ""),
        "gericht":            it.get("Gericht", ""),
        "dokumenttyp":        it.get("Dokumenttyp", "Text"),
        "geschaeftszahl":     it.get("Geschaeftszahl", ""),
        "normen":             "",
        "entscheidungsdatum": it.get("Entscheidungsdatum", ""),
        "ecli":               "",
        "rechtsgebiete":      "",
        "rechtssatznummern":  "",
        "schlagworte":        "",
        "beisaetze":          "",
        "anmerkung":          it.get("Anmerkung", ""),
        "entscheidungstexte": [],
        "n_entscheidungstexte": 0,
        "from_rechtssatz":    sorted(x for x in info["from_rs"] if x),
        "dokument_url":       url,
        "veroeffentlicht":    "",
        "geaendert":          "",
        "content_url_html":   f"https://www.ris.bka.gv.at/Dokumente/Justiz/{dn}/{dn}.html",
        "content_url_xml":    "",
        "content_url_pdf":    "",
        "full_text":          txt,
        "text_length":        len(txt),
        "matched_keywords":   sorted(info["keywords"]),
    }
    add_provenance(rec, "ris", "AT", "de", url or rec["content_url_html"], dn)
    return rec


def ris_keep_decision(rec):
    # civil, domestic decisions only. Strafrecht carries the *Entfremdung* homonym
    # (misappropriation of property); AUSL courts are EGMR case summaries republished
    # in RIS, which would duplicate the ECHR corpus and cross the jurisdiction boundary.
    rg = (rec.get("rechtsgebiete") or "").lower()
    if any(x in rg for x in RIS_EXCLUDE_RECHTSGEBIETE):
        return False
    if (rec.get("gericht") or "").startswith(RIS_EXCLUDE_GERICHT_PREFIX):
        return False
    return True


RIS_CHECKPOINT = "ris_fulltext_checkpoint.json"   # in DATA_DIR; resumes an interrupted run


def ris_text_cache():
    # reuse full text already on disk instead of re-fetching it: politeness first,
    # and it keeps an extend run from replacing good text with a failed re-fetch.
    cache = {}
    ckpt = DATA_DIR / RIS_CHECKPOINT
    if ckpt.exists():
        try:
            cache.update({k: v for k, v in json.loads(ckpt.read_text(encoding="utf-8")).items() if v})
            log.info(f"[RIS] resumed {len(cache)} texts from {RIS_CHECKPOINT}")
        except (ValueError, OSError):
            log.warning(f"[RIS] unreadable checkpoint {RIS_CHECKPOINT} — ignoring")
    path = out_path("ris")
    if RUN_MODE == "extend" and path.exists():
        for r in load_records(path):
            if r.get("id") and (r.get("full_text") or "") and r["id"] not in cache:
                cache[r["id"]] = r["full_text"]
    log.info(f"[RIS] full-text cache: {len(cache)} decisions already available")
    return cache


def ris_save_checkpoint(texts):
    (DATA_DIR / RIS_CHECKPOINT).write_text(
        json.dumps(texts, ensure_ascii=False), encoding="utf-8")


def import_ris_fulltext(keywords, year_from, year_to):
    # search the decision texts directly. The API default (Rechtssätze) cannot reach a
    # decision whose headnote omits the phrase, or one that was never headnoted.
    by_id = {}
    for kw in keywords:
        hits, docs = ris_search({"Applikation": RIS_APP, "Suchworte": kw,
                                 "Dokumenttyp.SucheInEntscheidungstexten": "true",
                                 "EntscheidungsdatumVon": f"{year_from}-01-01",
                                 "EntscheidungsdatumBis": f"{year_to}-12-31"})
        kept = dropped = 0
        for doc in docs:
            rec = ris_extract(doc)
            did = rec.get("id")
            if not did or rec.get("dokumenttyp") != "Text":
                continue
            yr = _ris_year(rec)
            if yr is not None and not (year_from <= yr <= year_to):
                continue
            if not ris_keep_decision(rec):
                dropped += 1
                continue
            if did in by_id:
                if kw not in by_id[did]["matched_keywords"]:
                    by_id[did]["matched_keywords"].append(kw)
                continue
            rec["matched_keywords"] = [kw]
            rec["match_route"] = ["fulltext"]
            by_id[did] = rec
            kept += 1
        log.info(f"[RIS] full text '{kw}': {hits} hits -> {kept} kept, "
                 f"{dropped} dropped (criminal/AUSL), running total {len(by_id)}")
    return by_id


def import_ris(keywords, year_from, year_to):
    by_id = {}
    for kw in keywords:
        total, candidates = ris_search({"Applikation": RIS_APP, "Suchworte": kw})
        log.info(f"[RIS] Rechtssatz '{kw}': {total} hits, {len(candidates)} retrieved")
        for doc in candidates:
            rec = ris_extract(doc)
            did = rec.get("id")
            if not did:
                continue
            yr = _ris_year(rec)
            if yr is not None and not (year_from <= yr <= year_to):
                continue
            if did in by_id:
                if kw not in by_id[did]["matched_keywords"]:
                    by_id[did]["matched_keywords"].append(kw)
                continue
            rec["matched_keywords"] = [kw]
            rec["match_route"] = ["rechtssatz" if rec.get("dokumenttyp") == "Rechtssatz"
                                  else "rechtssatz-link"]
            by_id[did] = rec

    records = list(by_id.values())
    text_cache = ris_text_cache()
    for i, rec in enumerate(records):
        txt = text_cache.get(rec.get("id")) or (ris_full_text(rec) if FETCH_FULL_TEXT else "")
        rec["full_text"] = txt
        rec["text_length"] = len(txt)
        add_provenance(rec, "ris", "AT", "de", rec.get("dokument_url", ""), rec.get("id"))
        if (i + 1) % 25 == 0:
            log.info(f"[RIS] Rechtssatz full text {i + 1}/{len(records)}")
    n_rs = len(records)
    log.info(f"[RIS] {n_rs} Rechtssätze collected")

    # expand the linked court decisions (Dokumenttyp "Text") into their own records
    if RIS_FETCH_DECISIONS:
        stubs = ris_collect_decision_stubs(records, year_from, year_to, RIS_DECISION_SCOPE)
        have = {r.get("id") for r in records}
        log.info(f"[RIS] expanding {len(stubs)} linked decisions (scope={RIS_DECISION_SCOPE})")
        done = 0
        for dn, info in stubs.items():
            if dn in have:
                continue
            rec = ris_decision_record(dn, info, text_cache.get(dn))
            rec["match_route"] = ["rechtssatz-link"]
            records.append(rec)
            done += 1
            if done % 25 == 0:
                log.info(f"[RIS] decision {done}/{len(stubs)}")

    # second route: the decision texts searched directly
    if RIS_FULLTEXT_SEARCH:
        n_before = len(records)
        have = {r.get("id") for r in records}
        ft = import_ris_fulltext(RIS_FULLTEXT_KEYWORDS, year_from, year_to)
        todo = [(did, rec) for did, rec in ft.items() if did not in have]
        log.info(f"[RIS] full-text route: {len(ft)} decisions, {len(todo)} not already held")
        texts = dict(text_cache)
        for i, (did, rec) in enumerate(todo):
            txt = text_cache.get(did) or (ris_full_text(rec) if FETCH_FULL_TEXT else "")
            rec["full_text"]   = txt
            rec["text_length"] = len(txt)
            add_provenance(rec, "ris", "AT", "de", rec.get("dokument_url", ""), did)
            records.append(rec)
            if txt:
                texts[did] = txt
            if (i + 1) % 50 == 0:
                if FETCH_FULL_TEXT:
                    ris_save_checkpoint(texts)
                log.info(f"[RIS] full-text decision {i + 1}/{len(todo)} "
                         f"({sum(1 for _, r in todo[:i + 1] if r.get('text_length'))} with text)")
        if FETCH_FULL_TEXT:
            ris_save_checkpoint(texts)
        log.info(f"[RIS] full-text route added {len(records) - n_before} decisions")

    log.info(f"[RIS] collected {len(records)} total records "
             f"({n_rs} Rechtssätze + {len(records) - n_rs} decisions)")
    return records


print("RIS functions ready")


RIS functions ready


## 4. ECHR — HUDOC via `echr_extractor`
Keeps the full HUDOC metadata column set (respondent, judgment date, articles,
violation/non-violation, conclusion, importance, application numbers, `scl`
case-citations). Date range is applied server-side via the HUDOC `kpdate` filter.

In [ ]:
ECHR_COLLECTIONS = ["JUDGMENTS", "COMMUNICATEDCASES", "DECISIONS"]
ECHR_LANG        = ["ENG"]

# HUDOC select fields: the extractor's meaningful defaults PLUS (verified live 2026-07-07):
#   kpthesaurus          - Registry-assigned legal-concept codes (structured topic metadata)
#   documentcollectionid - contains the FORMATION (COMMITTEE|CHAMBER|GRANDCHAMBER) = legal weight
#   introductiondate     - application filed -> proceedings-duration metrics become metadata
#   decisiondate, typedescription - extra decision metadata
ECHR_FIELDS = [
    "itemid", "applicability", "appno", "article", "conclusion", "docname", "doctype",
    "doctypebranch", "ecli", "importance", "judgementdate", "languageisocode",
    "originatingbody", "violation", "nonviolation", "extractedappno", "scl", "publishedby",
    "representedby", "respondent", "separateopinion", "sharepointid", "externalsources",
    "issue", "referencedate", "rulesofcourt",
    "kpthesaurus", "documentcollectionid", "typedescription", "introductiondate",
    "decisiondate",
]


def echr_build_link(fulltext, start_date, end_date):
    params = {
        "documentcollectionid2": ECHR_COLLECTIONS,
        "languageisocode": ECHR_LANG,
        "fulltext": [fulltext],
        "kpdate": [start_date, end_date],
    }
    return "https://hudoc.echr.coe.int/eng#" + json.dumps(params, ensure_ascii=False)


def _echr_df_records(df):
    if df is False or df is None or len(df) == 0:
        return []
    records = df.to_dict(orient="records")
    for r in records:
        for k, v in r.items():
            if v != v:        # NaN -> None
                r[k] = None
    return records


def import_echr(keywords, year_from, year_to):
    import echr_extractor as echr
    start_date, end_date = f"{year_from}-01-01", f"{year_to}-12-31"
    by_id, full_by_id = {}, {}

    for kw in keywords:
        link = echr_build_link(f'"{kw}"', start_date, end_date)   # exact-phrase fulltext
        log.info(f"[ECHR] '{kw}'")
        records, ft = [], {}
        try:
            if FETCH_FULL_TEXT:
                df, full_texts = echr.get_echr_extra(
                    link=link, count=ECHR_COUNT, language=ECHR_LANG, fields=ECHR_FIELDS,
                    save_file="n", verbose=False, progress_bar=True)
                records = _echr_df_records(df)
                # echr_extractor returns full_texts as a list of
                # {"item_id": ..., "ecli": ..., "full_text": ...} (not a dict)
                if isinstance(full_texts, list):
                    for it in full_texts:
                        iid, t = it.get("item_id"), it.get("full_text", "")
                        if iid and t:
                            ft[iid] = t
                elif isinstance(full_texts, dict):
                    ft = full_texts
            else:
                df = echr.get_echr(link=link, count=ECHR_COUNT, language=ECHR_LANG,
                                   fields=ECHR_FIELDS, save_file="n", verbose=False, progress_bar=True)
                records = _echr_df_records(df)
        except Exception as e:
            log.warning(f"[ECHR] query failed for '{kw}': {e}")
            continue

        for r in records:
            iid = r.get("itemid") or r.get("DocId") or r.get("WorkId")
            if not iid:
                continue
            if iid in by_id:
                if kw not in by_id[iid]["matched_keywords"]:
                    by_id[iid]["matched_keywords"].append(kw)
            else:
                r["matched_keywords"] = [kw]
                by_id[iid] = r
        for iid, t in ft.items():
            if t:
                full_by_id[iid] = t
        time.sleep(REQUEST_DELAY)

    records = list(by_id.values())
    for r in records:
        iid = r.get("itemid")
        t = full_by_id.get(iid, "")
        r["full_text"] = t
        r["text_length"] = len(t)
        lang = (r.get("languageisocode") or "ENG").lower()[:2]
        url = f"https://hudoc.echr.coe.int/eng?i={iid}"
        add_provenance(r, "echr", "Council of Europe", lang, url, iid)
    log.info(f"[ECHR] collected {len(records)} unique records")
    return records


print("ECHR functions ready")

ECHR functions ready


## 5. Swiss — entscheidsuche.ch (Elasticsearch)
Live ES scrape, German-filtered (`attachment.language = de`), date range applied
server-side. Full text is read from the ES-extracted `attachment.content` field
(reliable) rather than fetching and parsing the PDF. Captures title, abstract,
reference numbers, court hierarchy, canton and collection.

In [ ]:
SWISS_SEARCH   = "https://entscheidsuche.ch/_search.php"
SWISS_FALLBACK = "https://entscheidsuche.pansoft.de:9200/entscheidsuche-*/_search"
SWISS_FT       = "attachment.content"
SWISS_LANG     = "de"
SWISS_PAGE     = 50


def swiss_post(body, retries=3):
    for url in [SWISS_SEARCH, SWISS_FALLBACK]:
        for attempt in range(retries):
            try:
                time.sleep(REQUEST_DELAY if attempt == 0 else 5)
                r = session.post(url, json=body, timeout=30,
                                 headers={"Content-Type": "application/json"})
                if r.status_code == 200:
                    return r.json()
                if r.status_code in (429, 503):
                    time.sleep(30 * (attempt + 1))
                    continue
                break   # non-transient — try fallback url
            except (requests.exceptions.RequestException, ValueError):
                time.sleep(3 * attempt)
    return None


def _swiss_lang(d):
    # title/abstract come as {"de": "...", "fr": "...", ...}; prefer German
    if isinstance(d, dict):
        return d.get("de") or next(iter(d.values()), "")
    return d or ""


def _swiss_excluded(court_type):
    # court_type is hierarchy[1] e.g. 'ZH_VG'; check the part after the canton
    suffix = court_type.split("_", 1)[1] if "_" in court_type else court_type
    return suffix.upper() in SWISS_EXCLUDE_COURTS


def swiss_search(keyword, year_from, year_to):
    results, frm = [], 0
    start_date, end_date = f"{year_from}-01-01", f"{year_to}-12-31"
    while frm < SWISS_MAX:
        body = {
            "query": {"bool": {
                # match_phrase = exact phrase. Plain "match" ORs the tokens,
                # so a multi-word phrase pulls every doc containing "of"/"the"/
                # "child" etc. ("best interests of the child": 6426 hits vs 4).
                "must": [{"match_phrase": {SWISS_FT: keyword}}],
                "filter": [
                    {"term":  {"attachment.language": SWISS_LANG}},
                    {"range": {"date": {"gte": start_date, "lte": end_date}}},
                ],
            }},
            "size": SWISS_PAGE, "from": frm, "sort": [{"date": "desc"}],
            "_source": {"excludes": [SWISS_FT]},
        }
        data = swiss_post(body)
        if not data:
            break
        hits_meta = data.get("hits", {})
        hits = hits_meta.get("hits", [])
        total = hits_meta.get("total", {})
        total = total.get("value", 0) if isinstance(total, dict) else int(total or 0)
        if not hits:
            break
        for h in hits:
            src = h.get("_source", {})
            src.setdefault("id", h.get("_id", ""))
            results.append(src)
        frm += SWISS_PAGE
        if frm >= total:
            break
    return results


def swiss_full_text(sig):
    # second ES lookup for the extracted PDF text of one document
    data = swiss_post({"query": {"term": {"id": sig}}, "size": 1,
                       "_source": [SWISS_FT]})
    try:
        att = data["hits"]["hits"][0]["_source"].get("attachment", {})
        return att.get("content", "") or ""
    except (KeyError, IndexError, TypeError):
        return ""


def import_swiss(keywords, year_from, year_to):
    by_sig = {}
    for kw in keywords:
        log.info(f"[Swiss] '{kw}'")
        for src in swiss_search(kw, year_from, year_to):
            sig = src.get("id")
            if not sig:
                continue
            _hier = src.get("hierarchy") or []
            if SWISS_FAMILY_COURTS_ONLY and _swiss_excluded(_hier[1] if len(_hier) > 1 else ""):
                continue
            if sig in by_sig:
                if kw not in by_sig[sig]["matched_keywords"]:
                    by_sig[sig]["matched_keywords"].append(kw)
                continue
            hierarchy = src.get("hierarchy") or []
            att = src.get("attachment") or {}
            datum = str(src.get("date", ""))[:10]
            by_sig[sig] = {
                "Signatur":    sig,
                "Spider":      "/".join(hierarchy),
                "hierarchy":   hierarchy,
                "canton":      src.get("canton") or (hierarchy[0] if hierarchy else ""),
                "court_type":  hierarchy[1] if len(hierarchy) > 1 else "",
                "Sprache":     att.get("language", ""),
                "Datum":       datum,
                "year":        (int(datum[:4]) if datum[:4].isdigit() else None),
                "title":       _swiss_lang(src.get("title")),
                "abstract":    _swiss_lang(src.get("abstract")),
                "reference":   src.get("reference") or [],
                "collection":  src.get("source", ""),
                "scrapedate":  src.get("scrapedate", ""),
                "content_url":  att.get("content_url", ""),
                "content_type": att.get("content_type", ""),
                "matched_keywords": [kw],
            }
        time.sleep(REQUEST_DELAY)

    records = list(by_sig.values())
    for i, rec in enumerate(records):
        txt = swiss_full_text(rec["Signatur"]) if FETCH_FULL_TEXT else ""
        rec["content"] = txt
        rec["text_length"] = len(txt)
        add_provenance(rec, "swiss", "CH", rec.get("Sprache") or "de",
                       rec.get("content_url", ""), rec["Signatur"])
        if (i + 1) % 25 == 0:
            log.info(f"[Swiss] full text {i + 1}/{len(records)}")
    log.info(f"[Swiss] collected {len(records)} unique records")
    return records


print("Swiss functions ready")

Swiss functions ready


## 6. Open Legal Data (Germany) — de.openlegaldata.io
Search has no date filter, so the year range is applied in Python. Full case fetched
via the 2-step slug -> id -> content pattern; content stripped to plain text.
Captures the full court object (jurisdiction/level/name/city/state), file number,
ECLI, decision type, created/updated dates and the original source URL.

In [ ]:
DE_API = "https://de.openlegaldata.io/api"


def de_get(url, params=None, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.get(url, params=params, timeout=20)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.HTTPError:
            if r.status_code == 429:
                time.sleep(30 * (attempt + 1))
            elif r.status_code >= 500:
                time.sleep(5 * (attempt + 1))
            else:
                return None
        except (requests.exceptions.RequestException, ValueError):
            time.sleep(3 * (attempt + 1))
    return None


def _de_in_range(date_str, year_from, year_to):
    try:
        y = int(str(date_str)[:4])
    except (ValueError, TypeError):
        return False
    return year_from <= y <= year_to


def de_search(keyword, year_from, year_to):
    results = []
    url = f"{DE_API}/cases/search/"
    params = {"text": keyword, "format": "json"}
    page = 0
    start_date = f"{year_from}-01-01"
    while url and page < DE_MAX_PAGES:
        data = de_get(url, params)
        params = None    # subsequent calls follow the full `next` url
        page += 1
        if not data:
            break
        page_results = data.get("results", [])
        if isinstance(page_results, dict):
            page_results = [page_results]
        for r in page_results:
            if DE_ORDINARY_ONLY and r.get("court_jurisdiction") in DE_EXCLUDE_JURIS:
                continue
            if _de_in_range(r.get("date", ""), year_from, year_to):
                results.append(r)
        # results are date-descending: stop once a whole page predates the range
        dates = [r.get("date", "") for r in page_results if r.get("date")]
        if dates and max(dates) < start_date:
            break
        url = data.get("next")
    return results


def de_fetch_case(slug, fetch_text=True):
    ld = de_get(f"{DE_API}/cases/", params={"slug": slug, "format": "json"})
    if not ld or not ld.get("results"):
        return None
    meta = ld["results"][0]
    cid = meta.get("id")
    detail = de_get(f"{DE_API}/cases/{cid}/", params={"format": "json"}) if fetch_text else {}
    detail = detail or {}
    court = meta.get("court") or detail.get("court") or {}
    if isinstance(court, str):
        court = {"slug": court, "name": court}
    content_text = html_to_text(detail.get("content", ""))
    date_str = meta.get("date", "")
    return {
        "id":                 cid,
        "slug":               slug,
        "court_name":         court.get("name", ""),
        "court_slug":         court.get("slug", ""),
        "court_level":        court.get("level_of_appeal", ""),
        "court_jurisdiction": court.get("jurisdiction"),
        "court_city":         court.get("city"),
        "court_state":        court.get("state"),
        "file_number":        meta.get("file_number", ""),
        "date":               date_str,
        "year":               (int(str(date_str)[:4]) if str(date_str)[:4].isdigit() else None),
        "decision_type":      meta.get("type", detail.get("type", "")),
        "ecli":               meta.get("ecli", detail.get("ecli", "")),
        "created_date":       meta.get("created_date", detail.get("created_date", "")),
        "updated_date":       meta.get("updated_date", detail.get("updated_date", "")),
        "original_source_url": meta.get("source_url", detail.get("source_url", "")),
        "content":            content_text,
        "text_length":        len(content_text),
    }


def import_germany(keywords, year_from, year_to):
    slug_kw = {}
    for kw in keywords:
        log.info(f"[Germany] '{kw}'")
        for r in de_search(kw, year_from, year_to):
            slug = r.get("slug")
            if slug:
                slug_kw.setdefault(slug, set()).add(kw)
        time.sleep(REQUEST_DELAY)

    log.info(f"[Germany] {len(slug_kw)} unique slugs; fetching cases")
    records = []
    for i, slug in enumerate(slug_kw):
        rec = de_fetch_case(slug, fetch_text=FETCH_FULL_TEXT)
        if rec is None:
            # stub so a re-run can still see / retry the slug
            rec = {"id": None, "slug": slug, "content": "", "text_length": 0,
                   "court_name": "", "court_slug": "", "court_level": "",
                   "court_jurisdiction": None, "court_city": None, "court_state": None,
                   "file_number": "", "date": "", "year": None, "decision_type": "",
                   "ecli": "", "created_date": "", "updated_date": "",
                   "original_source_url": ""}
        rec["matched_keywords"] = sorted(slug_kw[slug])
        add_provenance(rec, "germany", "DE", "de",
                       f"https://de.openlegaldata.io/case/{slug}", slug)
        records.append(rec)
        if (i + 1) % 25 == 0:
            log.info(f"[Germany] {i + 1}/{len(slug_kw)}")
    log.info(f"[Germany] collected {len(records)} unique records")
    return records


print("Germany functions ready")

Germany functions ready


## 7. Orchestrator — run every source per `RUN_MODE`
Each source is imported independently into its own file. `new` archives any existing
file then imports fresh; `extend` loads the existing file, fetches the delta and
merges idempotently by `stable_id`.

In [ ]:
SOURCE_REGISTRY = {
    "ris":     import_ris,
    "echr":    import_echr,
    "swiss":   import_swiss,
    "germany": import_germany,
}

summary = {}
for src in SOURCES:
    if src not in SOURCE_REGISTRY:
        log.warning(f"unknown source '{src}' — skipping")
        continue
    kw_for_src = KEYWORDS.get(src, [])
    if not kw_for_src:
        log.warning(f"no keywords configured for '{src}' — skipping")
        continue
    import_fn = SOURCE_REGISTRY[src]
    path = out_path(src)
    log.info("=" * 64)
    log.info(f"SOURCE: {src}   MODE: {RUN_MODE}   FILE: {path.name}   keywords: {kw_for_src}")
    log.info("=" * 64)

    if RUN_MODE == "new":
        archive_if_exists(path)
        records = import_fn(kw_for_src, YEAR_FROM, YEAR_TO)
    else:  # extend
        if not path.exists():
            raise FileNotFoundError(
                f"extend mode: no existing file {path}. "
                f"Run RUN_MODE='new' for topic '{TOPIC}' first.")
        existing = load_records(path)
        fetched  = import_fn(kw_for_src, YEAR_FROM, YEAR_TO)
        records  = merge_by_id(existing, fetched)

    save_records(path, records)
    with_text = sum(1 for r in records if (r.get("text_length") or 0) > 100)
    summary[src] = (len(records), with_text, str(path))

print("\n" + "=" * 64)
print(f"DONE  mode={RUN_MODE}  topic={TOPIC}  years={YEAR_FROM}-{YEAR_TO}")
print("=" * 64)
for src, (n, wt, p) in summary.items():
    print(f"  {src:9s}: {n:5d} records  ({wt} with full text)  -> {p}")


                                                                  
DONE  mode=new  topic=parental_alienation  years=2000-2025
  ris      :   548 records  (548 with full text)  -> ../data/ris_parental_alienation.json
  echr     :  1116 records  (1116 with full text)  -> ../data/echr_parental_alienation.json
  swiss    :  2031 records  (2031 with full text)  -> ../data/swiss_parental_alienation.json


## 8. Quick verification (read-only)
Confirms each file is loadable, every record has a stable id + provenance, and ids are
unique. No NLP — purely a sanity check on what was written.

In [ ]:
PROV = ["stable_id", "source", "jurisdiction", "lang", "source_url", "retrieved_date"]

for src in SOURCES:
    path = out_path(src)
    if not path.exists():
        print(f"{src:9s}: (no file)")
        continue
    recs = load_records(path)
    ids = [r.get("stable_id") for r in recs]
    n_ids = sum(1 for i in ids if i)
    n_uniq = len(set(i for i in ids if i))
    n_prov = sum(1 for r in recs if all(r.get(k) not in (None, "") for k in PROV))
    n_text = sum(1 for r in recs if (r.get("text_length") or 0) > 100)
    flag = "OK" if (n_ids == len(recs) == n_uniq) else "CHECK"
    print(f"{src:9s}: {len(recs):5d} recs | ids {n_ids}/{len(recs)} uniq {n_uniq} "
          f"| provenance {n_prov}/{len(recs)} | full text {n_text}  [{flag}]")
    if recs:
        print(f"           sample stable_id={recs[0].get('stable_id')!r} "
              f"url={recs[0].get('source_url')!r}")

ris      :   548 recs | ids 548/548 uniq 548 | provenance 548/548 | full text 548  [OK]
           sample stable_id='JJR_19811203_OGH0002_0130OS00176_8000000_002' url='https://www.ris.bka.gv.at/Dokument.wxe?Abfrage=Justiz&Dokumentnummer=JJR_19811203_OGH0002_0130OS00176_8000000_002'
echr     :  1116 recs | ids 1116/1116 uniq 1116 | provenance 1116/1116 | full text 1116  [OK]
           sample stable_id='001-105026' url='https://hudoc.echr.coe.int/eng?i=001-105026'
swiss    :  2031 recs | ids 2031/2031 uniq 2031 | provenance 2031/2031 | full text 2031  [OK]
           sample stable_id='ZH_OG_001_LZ250034_2025-12-30' url='https://entscheidsuche.ch/docs/ZH_Obergericht/ZH_OG_001_LZ250034_2025-12-30.pdf'
